In [109]:
import torch
import numpy as np
import random
from torch.utils.data import DataLoader
import os
import urllib
import zipfile
import lxml.etree
import re
from collections import Counter

In [110]:
pip install --upgrade --force-reinstall numpy pandas scikit-learn

  Using cached numpy-2.2.5-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pandas-2.2.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (89 kB)
  Using cached scikit_learn-1.6.1-cp311-cp311-macosx_12_0_arm64.whl.metadata (31 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.15.2-cp311-cp311-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-2.2.5-cp311-cp311-macosx_14_0_arm64.whl (5.4 MB)
Using cached pandas-2.2.3-cp311-cp311-macosx_11_0_arm64.whl (11.3 MB)
Using cached scikit_learn-1.6.1-cp311-cp311-macosx_12_0_arm64.whl (11.1 MB)
Using cached joblib-1.4.2-py3-none-any.whl (301 kB)
U

In [111]:
if not os.path.isfile('ted_en-20160408.xml'):
    urllib.request.urlretrieve("https://github.com/oxford-cs-deepnlp-2017/practical-1/blob/master/ted_en-20160408.xml?raw=true", filename="ted_en-20160408.xml")

In [112]:
doc = lxml.etree.parse('ted_en-20160408.xml')
input_text = doc.xpath('//content/text()')
label = doc.xpath('//head/keywords/text()')
del doc
len(input_text)
texts = [text for text in input_text]

In [113]:
texts_labels=zip(texts,label)
texts = [text_label for text_label in texts_labels if len(text_label[0]) > 500]
print('number of text greater than 500 words are:',len(texts))

number of text greater than 500 words are: 2076


In [114]:
texts,labels=zip(*texts)
print(labels[:10])

('talks, business, creativity, curiosity, goal-setting, innovation, motivation, potential, success, work', 'talks, Planets, TEDx, bacteria, biology, engineering, environment, evolution, exploration, future, innovation, intelligence, microbiology, nature, potential, science', 'talks, Debate, Guns, activism, big problems, children, choice, community, future, goal-setting, government, law, leadership, marketing, parenting, policy, social change, violence', 'talks, Brazil, Slavery, art, beauty, community, creativity, culture, design, global issues, humanity, identity, photography, race, social change, society, visualizations', 'talks, NASA, communication, computers, creativity, design, engineering, exploration, future, innovation, interface design, invention, microsoft, potential, prediction, product design, technology, visualizations', 'talks, Africa, Internet, community, democracy, development, future, government, identity, leadership, politics, potential', 'talks, ancient world, animals

In [115]:
tokens = [words for text in texts for words in text.split()]
words_count = Counter(tokens)
words_least_common = set([word for word,count in words_count.most_common() if count==1])

In [116]:
texts = [[word for word in text.split() if word not in words_least_common]for text in texts]

In [117]:
all_labels=[]
for i,keyword in enumerate(labels):
    key = keyword.split(', ')
    all_labels+=key

all_labels_set = set(all_labels)

In [118]:
label2id = {label: idx for idx, label in enumerate(all_labels_set)}
id2label = {idx: label for label, idx in label2id.items()}

In [119]:
count_labels=Counter(all_labels)
label_count = [word_count for word_count in count_labels.most_common()]
label_count

[('talks', 2076),
 ('TED Conference', 698),
 ('technology', 595),
 ('culture', 451),
 ('science', 447),
 ('global issues', 428),
 ('design', 352),
 ('TEDx', 321),
 ('business', 289),
 ('entertainment', 262),
 ('arts', 183),
 ('education', 159),
 ('politics', 157),
 ('health', 155),
 ('creativity', 147),
 ('art', 135),
 ('economics', 119),
 ('medicine', 118),
 ('biology', 117),
 ('TED Fellows', 114),
 ('brain', 109),
 ('music', 102),
 ('cities', 101),
 ('social change', 100),
 ('invention', 99),
 ('storytelling', 97),
 ('environment', 96),
 ('activism', 89),
 ('children', 87),
 ('health care', 87),
 ('innovation', 86),
 ('future', 83),
 ('women', 83),
 ('war', 83),
 ('history', 82),
 ('psychology', 81),
 ('photography', 80),
 ('animals', 80),
 ('collaboration', 76),
 ('humor', 76),
 ('communication', 73),
 ('Africa', 71),
 ('computers', 69),
 ('architecture', 66),
 ('exploration', 63),
 ('society', 63),
 ('oceans', 60),
 ('nature', 59),
 ('performance', 59),
 ('happiness', 57),
 ('physi

In [120]:
labels_indices = []
for keyword in labels:
    first_label = keyword.split(', ')[1]  # Take only the first label
    labels_indices.append(label2id[first_label])
# Check the labels indices for the first few samples
print(labels_indices[:10])  # This will print the indices of the labels for the first 10 samples

[88, 65, 97, 165, 116, 365, 74, 264, 37, 313]


In [121]:
tokens.append('<UNK>')
tokens.append('<PAD>')

In [122]:
vocab = list(set(tokens))

In [123]:
print('size of vocabulary:',len(vocab))
id2word = dict(enumerate(vocab))
word2id = dict((val,key) for (key,val) in id2word.items())

size of vocabulary: 142892


In [124]:
# Stripping Text to fall within length of 500; incase if it is shorter then padd with '<UNK>'
length = 500 #sentence length
stripped_text = []#np.zeros((len(texts),length)
for i,text in enumerate(texts):
    inputs = []
    if len(text) >= 500:
        inputs.extend(text[:500])
    else:
        extra_length = 500-len(text)
        extra = ['<PAD>']*extra_length
        word_with_extra = text + extra
        inputs.extend(word_with_extra)
    stripped_text.append(inputs) 

In [125]:
stripped_length = len(stripped_text)
print(stripped_length)

2076


In [126]:
inputs = []
text_ids = []
for text in stripped_text:
    for word in text:
        i = word2id[word]
        inputs.append(i)
    text_ids.append(inputs)
    inputs = []

In [127]:
data = list(zip(text_ids, labels_indices))  # Each item is a tuple (text, [label_indices]

In [128]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80-20 split)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Separate the inputs (text) and labels for train and test data
train_texts, train_labels = zip(*train_data)
test_texts, test_labels = zip(*test_data)

# ✅ Convert everything to tensors for PyTorch (single label per example)
train_texts_tensor = torch.tensor(train_texts, dtype=torch.long)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)  # <— FIXED

test_texts_tensor = torch.tensor(test_texts, dtype=torch.long)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)  

In [129]:
# 👇 New special tokens for conditioning
SPECIAL_TOKENS = ['<|label|>', '<|text|>']

def format_for_gpt2(text_ids, label_id):
    label = id2label[label_id]
    text_words = [id2word[i] for i in text_ids if id2word[i] not in ('<PAD>', '<UNK>')]
    text_str = ' '.join(text_words)
    return f"<|label|> {label} <|text|> {text_str}"

formatted_texts = [format_for_gpt2(text, label) for text, label in zip(train_texts, train_labels)]
formatted_val_texts = [format_for_gpt2(text, label) for text, label in zip(test_texts, test_labels)]


In [130]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torch.utils.data import Dataset

# Load and expand GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'additional_special_tokens': SPECIAL_TOKENS})

class TEDDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.inputs = tokenizer(texts, return_tensors='pt', truncation=True,
                                padding='max_length', max_length=max_length)
    
    def __len__(self):
        return len(self.inputs['input_ids'])
    
    def __getitem__(self, idx):
        return {
            'input_ids': self.inputs['input_ids'][idx],
            'attention_mask': self.inputs['attention_mask'][idx],
            'labels': self.inputs['input_ids'][idx]
        }

tokenizer.pad_token = tokenizer.eos_token

train_dataset = TEDDataset(formatted_texts, tokenizer)
val_dataset = TEDDataset(formatted_val_texts, tokenizer)


In [131]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))  # accommodate new tokens


Embedding(50259, 768)

In [132]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50259, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50259, bias=False)
)

In [133]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2)


In [134]:
optimizer = AdamW(model.parameters(), lr=5e-5)

from transformers import get_scheduler
num_training_steps = len(train_loader) * 3  # 3 epochs

lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer,
    num_warmup_steps=100, num_training_steps=num_training_steps
)


In [135]:
''' epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} avg training loss: {avg_loss:.4f}") '''


' epochs = 3\n\nfor epoch in range(epochs):\n    model.train()\n    total_loss = 0\n    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")\n    \n    for batch in loop:\n        input_ids = batch[\'input_ids\'].to(device)\n        attention_mask = batch[\'attention_mask\'].to(device)\n        labels = batch[\'labels\'].to(device)\n\n        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)\n        loss = outputs.loss\n\n        loss.backward()\n        optimizer.step()\n        lr_scheduler.step()\n        optimizer.zero_grad()\n\n        total_loss += loss.item()\n        loop.set_postfix(loss=loss.item())\n\n    avg_loss = total_loss / len(train_loader)\n    print(f"\nEpoch {epoch+1} avg training loss: {avg_loss:.4f}") '

In [136]:
model.load_state_dict(torch.load("text_generation_model3.pth", weights_only=True))
model.eval()
def generate_text(model, tokenizer, label_name, max_new_tokens=300, device='cuda' if torch.cuda.is_available() else 'cpu'):
    model.eval()
    model.to(device)

    # Start prompt with special conditioning
    prompt = f"<|label|> {label_name} <|text|>"
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Generate continuation
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode and print
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return generated_text

torch.save(model.state_dict(), 'text_generation_model3.pth')

In [156]:
label = "meat"
generated = generate_text(model, tokenizer, label)
print("\n"+generated)



 meat  I think that animals need to be treated with respect, respect and compassion. I think that we can take them to places where we can and that will allow them to live as they please. I think that animal care should be a fundamental part of our life. And I think it should be an integral part of the human experience. I think it should be part of our daily lives. I think there should be compassion, empathy and compassion in many of us. And I think it should be a part of human beings. I think our culture should be compassionate to animals. But I think it is not enough to be compassionate to animals. It is important to be compassionate to animals. And I think that we need to go beyond just being compassionate. We need to go beyond just being compassionate to animals. We need to create a world where we don't have to be. And I think that we can be compassionate to animals in a way that I think is going to allow us to become, in many ways, better. And I think there is a lot to be done. I 